# Pendulo simples: Definicao de derivada, Euler e RK4

Neste notebook, o foco e didatico em **posicao angular**.
A ultima celula mostra no mesmo bloco:
- o aparato oscilando (painel esquerdo);
- as 3 curvas de posicao sendo formadas (painel direito).
    


## Aparato experimental usado

- massa puntiforme ligada a uma haste rigida de comprimento `L`;
- pivo fixo;
- movimento no plano sem amortecimento.

Equacao modelada:
`theta'' + (g/L) * sin(theta) = 0`
    


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

plt.rcParams['animation.html'] = 'jshtml'
    


In [ ]:
# BLOCO 1 - Parametros
g = 9.81
L = 1.0
theta0 = np.deg2rad(20.0)
omega0 = 0.0
dt = 0.01
t_final = 20.0
t = np.arange(0.0, t_final + dt, dt)


def mostrar_amostra(nome, t, theta, n=10):
    x = L * np.sin(theta)
    print(f"\nSaida numerica - {nome}")
    print(" i   t(s)    theta(rad)      x(m)")
    for i in range(min(n, len(t))):
        print(f"{i:2d}  {t[i]:6.3f}   {theta[i]:10.6f}   {x[i]:8.5f}")
    


In [ ]:
# BLOCO 2 - Definicao de derivada (diferenca finita de 2a ordem)
def simular_definicao_derivada(t, g, L, theta0, omega0):
    dt = t[1] - t[0]
    n = len(t)
    theta = np.zeros(n)

    theta[0] = theta0
    alpha0 = -(g / L) * np.sin(theta0)
    theta[1] = theta0 + omega0 * dt + 0.5 * alpha0 * dt**2

    for i in range(1, n - 1):
        alpha_i = -(g / L) * np.sin(theta[i])
        theta[i + 1] = 2 * theta[i] - theta[i - 1] + alpha_i * dt**2

    return theta


# BLOCO 3 - Euler
def simular_euler(t, g, L, theta0, omega0):
    dt = t[1] - t[0]
    n = len(t)
    theta = np.zeros(n)
    omega = np.zeros(n)

    theta[0] = theta0
    omega[0] = omega0

    for i in range(n - 1):
        alpha = -(g / L) * np.sin(theta[i])
        omega[i + 1] = omega[i] + alpha * dt
        theta[i + 1] = theta[i] + omega[i] * dt

    return theta


# BLOCO 4 - RK4
def derivadas(theta, omega, g, L):
    return omega, -(g / L) * np.sin(theta)


def simular_rk4(t, g, L, theta0, omega0):
    dt = t[1] - t[0]
    n = len(t)
    theta = np.zeros(n)
    omega = np.zeros(n)

    theta[0] = theta0
    omega[0] = omega0

    for i in range(n - 1):
        th, om = theta[i], omega[i]

        k1_th, k1_om = derivadas(th, om, g, L)
        k2_th, k2_om = derivadas(th + 0.5 * dt * k1_th, om + 0.5 * dt * k1_om, g, L)
        k3_th, k3_om = derivadas(th + 0.5 * dt * k2_th, om + 0.5 * dt * k2_om, g, L)
        k4_th, k4_om = derivadas(th + dt * k3_th, om + dt * k3_om, g, L)

        theta[i + 1] = th + (dt / 6.0) * (k1_th + 2 * k2_th + 2 * k3_th + k4_th)
        omega[i + 1] = om + (dt / 6.0) * (k1_om + 2 * k2_om + 2 * k3_om + k4_om)

    return theta
    


In [ ]:
# BLOCO 5 - Simulacoes
theta_def = simular_definicao_derivada(t, g, L, theta0, omega0)
theta_euler = simular_euler(t, g, L, theta0, omega0)
theta_rk4 = simular_rk4(t, g, L, theta0, omega0)

mostrar_amostra('Definicao de derivada', t, theta_def)
mostrar_amostra('Euler', t, theta_euler)
mostrar_amostra('RK4', t, theta_rk4)
    


## Mesma celula: aparato oscilando + 3 curvas de posicao se formando

No painel esquerdo, o aparato do pendulo e animado usando RK4.
No painel direito, as tres curvas (`Def. derivada`, `Euler`, `RK4`) aparecem em tempo real.
    


In [ ]:
def animar_aparato_e_curvas(t, L, theta_aparato, theta_def, theta_euler, theta_rk4, titulo='Pendulo'):
    max_frames = 420
    passo = max(1, len(t) // max_frames)

    t_anim = t[::passo]
    th_app = theta_aparato[::passo]
    th_def = theta_def[::passo]
    th_eul = theta_euler[::passo]
    th_rk = theta_rk4[::passo]

    fig, (ax_p, ax_g) = plt.subplots(1, 2, figsize=(12, 4.8))
    fig.suptitle(titulo)

    # Painel esquerdo: aparato
    ax_p.set_aspect('equal')
    ax_p.set_xlim(-1.2 * L, 1.2 * L)
    ax_p.set_ylim(-1.2 * L, 0.2 * L)
    ax_p.set_title('Aparato experimental: pendulo')
    ax_p.plot(0, 0, 'ko', ms=6)
    haste, = ax_p.plot([], [], lw=2, color='tab:blue')
    bob, = ax_p.plot([], [], 'o', ms=10, color='tab:red')

    # Painel direito: 3 curvas se formando
    amp = max(np.max(np.abs(np.concatenate([th_def, th_eul, th_rk]))) * 1.2, 0.2)
    ax_g.set_xlim(0, t[-1])
    ax_g.set_ylim(-amp, amp)
    ax_g.set_title('theta(t): Def. derivada, Euler e RK4')
    ax_g.set_xlabel('tempo (s)')
    ax_g.set_ylabel('theta (rad)')
    ax_g.grid(alpha=0.3)

    linha_def, = ax_g.plot([], [], color='black', linestyle='--', linewidth=2.0, label='Def. derivada', zorder=4)
    linha_eul, = ax_g.plot([], [], color='tab:orange', linewidth=1.8, label='Euler', zorder=3)
    linha_rk, = ax_g.plot([], [], color='tab:green', linewidth=1.8, label='RK4', zorder=2)

    pt_def, = ax_g.plot([], [], 'o', color='black', ms=4)
    pt_eul, = ax_g.plot([], [], 'o', color='tab:orange', ms=4)
    pt_rk, = ax_g.plot([], [], 'o', color='tab:green', ms=4)
    ax_g.legend(loc='upper right')

    def init():
        haste.set_data([], [])
        bob.set_data([], [])
        linha_def.set_data([], [])
        linha_eul.set_data([], [])
        linha_rk.set_data([], [])
        pt_def.set_data([], [])
        pt_eul.set_data([], [])
        pt_rk.set_data([], [])
        return haste, bob, linha_def, linha_eul, linha_rk, pt_def, pt_eul, pt_rk

    def update(i):
        th = th_app[i]
        x = L * np.sin(th)
        y = -L * np.cos(th)
        haste.set_data([0, x], [0, y])
        bob.set_data([x], [y])

        tt = t_anim[: i + 1]
        yd = th_def[: i + 1]
        ye = th_eul[: i + 1]
        yr = th_rk[: i + 1]

        linha_def.set_data(tt, yd)
        linha_eul.set_data(tt, ye)
        linha_rk.set_data(tt, yr)

        pt_def.set_data([tt[-1]], [yd[-1]])
        pt_eul.set_data([tt[-1]], [ye[-1]])
        pt_rk.set_data([tt[-1]], [yr[-1]])
        return haste, bob, linha_def, linha_eul, linha_rk, pt_def, pt_eul, pt_rk

    ani = FuncAnimation(fig, update, frames=len(t_anim), init_func=init, interval=30, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())


display(animar_aparato_e_curvas(t, L, theta_rk4, theta_def, theta_euler, theta_rk4, titulo='Pendulo: aparato + curvas'))
    
